# Phase 2 — Pseudo-labels from reports

Hybrid report labeling (rules + optional LLM) → train ConvNeXt-Tiny MIL on 4,407 studies.

- OOF metric computed on **58 labeled** studies only (compare to Phase 1 baseline 0.5295).
- Test inference is **image-only** (no reports).
- Pretrained backbone: set `cfg["model"]["pretrained_weights"]` / `variant` in export script.
- Daily: Cursor → Kaggle Jupyter Server ([docs/KAGGLE.md](../docs/KAGGLE.md)).
- Submit: `python scripts/push_kaggle_kernel.py phase2` → Save & Run All.

## Setup

In [19]:
from __future__ import annotations

import importlib
import json
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch


def _src() -> Path:
    work = Path("/kaggle/working")
    work_src = work / "src"
    mount = Path("/kaggle/input/rsna-knee-code")
    mount_src = mount / "src"
    reports = work_src / "rsna_knee" / "reports"
    if work.is_dir() and (mount_src / "rsna_knee").is_dir():
        need_copy = not (work_src / "rsna_knee").is_dir()
        need_overlay = (work_src / "rsna_knee").is_dir() and not reports.is_dir()
        if need_copy:
            shutil.copytree(mount_src, work_src)
        elif need_overlay:
            shutil.copytree(mount_src, work_src, dirs_exist_ok=True)
        if (need_copy or need_overlay) and (mount / "configs").is_dir():
            shutil.copytree(mount / "configs", work / "configs", dirs_exist_ok=True)
    if (work_src / "rsna_knee").is_dir():
        if not reports.is_dir():
            raise FileNotFoundError(
                "rsna_knee.reports missing under /kaggle/working/src. "
                "Run: python scripts/sync_code_to_jupyter.py"
            )
        return work_src
    for root in (Path.cwd(), Path.cwd().parent):
        src = root / "src"
        if (src / "rsna_knee").is_dir():
            return src
    raise FileNotFoundError("Run: python scripts/sync_code_to_jupyter.py")


def _refresh_rsna_knee(src: Path) -> None:
    """Drop bytecode + in-memory modules so a re-run picks up a just-synced src/."""
    for cache in src.rglob("__pycache__"):
        shutil.rmtree(cache, ignore_errors=True)
    importlib.invalidate_caches()
    for key in list(sys.modules):
        if key == "rsna_knee" or key.startswith("rsna_knee."):
            del sys.modules[key]


_SRC = _src()
_src_str = str(_SRC)
sys.path = [_src_str] + [p for p in sys.path if p != _src_str]
_refresh_rsna_knee(_SRC)
REPO_ROOT = _SRC.parent

from rsna_knee.constants import TARGET_LABELS
from rsna_knee.data import load_test_table, predictions_to_submission
from rsna_knee.data.schema import labels_present_mask, load_train_table
from rsna_knee.models.weights import resolve_pretrained_weights
from rsna_knee.reports import evaluate_labeler, generate_pseudo_labels, report_eda_summary
from rsna_knee.reports.rules import RuleLabeler
from rsna_knee.training import macro_roc_auc, predict_test_ensemble, run_phase2_training
from rsna_knee.utils.config import load_config
from rsna_knee.utils.paths import default_data_root, is_kaggle_kernel

ON_KAGGLE = is_kaggle_kernel()
CFG_NAME = "kaggle_phase2" if ON_KAGGLE else "phase2"
cfg = load_config(CFG_NAME)
DATA_ROOT = default_data_root()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR = Path(cfg["paths"]["output_dir"])
if not OUT_DIR.is_absolute():
    OUT_DIR = REPO_ROOT / OUT_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Environment : {'Kaggle' if ON_KAGGLE else 'local'}")
print(f"Data root   : {DATA_ROOT}")
print(f"Package     : {_SRC}")
print(f"Config      : {CFG_NAME}")

Environment : Kaggle
Data root   : /kaggle/input/competitions/rsna-knee-abnormality-detection
Package     : /kaggle/working/src
Config      : kaggle_phase2


## 1. Report EDA

In [2]:
eda = report_eda_summary(DATA_ROOT)
print(json.dumps(eda, indent=2))
display(pd.DataFrame(eda["language_distribution"]))
display(pd.DataFrame(eda["keyword_hit_rates_labeled"]))

{
  "n_studies": 4407,
  "n_labeled": 58,
  "length_stats": {
    "n_reports": 4407.0,
    "chars_mean": 1097.905604719764,
    "chars_median": 977.0,
    "chars_p95": 2452.7,
    "words_mean": 147.54299977308827,
    "words_median": 129.0
  },
  "language_distribution": [
    {
      "language": "english",
      "count": 1750,
      "fraction": 0.3970955298388927
    },
    {
      "language": "other",
      "count": 1506,
      "fraction": 0.3417290673927842
    },
    {
      "language": "spanish",
      "count": 742,
      "fraction": 0.16836850465169048
    },
    {
      "language": "german",
      "count": 409,
      "fraction": 0.09280689811663263
    }
  ],
  "keyword_hit_rates_labeled": [
    {
      "label": "joint_effusion",
      "hit_rate": 0.5172413793103449,
      "accuracy": 0.5862068965517241,
      "n": 58
    },
    {
      "label": "acl_tear",
      "hit_rate": 0.4482758620689655,
      "accuracy": 0.603448275862069,
      "n": 58
    },
    {
      "label": "mcl_t

,language,count,fraction
0,english,1750,0.397096
1,other,1506,0.341729
2,spanish,742,0.168369
3,german,409,0.092807


,label,hit_rate,accuracy,n
0,joint_effusion,0.517241,0.586207,58
1,acl_tear,0.448276,0.603448,58
2,mcl_tear,0.448276,0.724138,58
3,medial_meniscus_injury,0.413793,0.586207,58
4,lateral_meniscus_injury,0.396552,0.689655,58
5,bone_contusion,0.310345,0.689655,58
6,fracture,0.189655,0.810345,58
7,patellofemoral_osteoarthritis,0.172414,0.534483,58
8,bakers_cyst,0.155172,0.844828,58
9,synovitis,0.155172,0.586207,58


## 2. Validate rule labeler on 58 labeled studies

In [4]:
rule_metrics, rule_macro_f1 = evaluate_labeler(RuleLabeler(), DATA_ROOT)
print(f"Rule labeler macro F1: {rule_macro_f1:.3f}")
display(rule_metrics.sort_values("f1", ascending=False))

Rule labeler macro F1: 0.394


,label,precision,recall,f1,auc,support_pos,support_neg
3,lateral_meniscus_injury,0.619048,0.565217,0.590909,0.668323,23,35
7,joint_effusion,0.761905,0.457143,0.571429,0.619876,35,23
11,fracture,1.000000,0.388889,0.560000,0.694444,18,40
0,acl_tear,0.518519,0.583333,0.549020,0.600490,24,34
10,bone_contusion,0.529412,0.473684,0.500000,0.634278,19,39
1,mcl_tear,0.333333,0.777778,0.466667,0.746032,9,49
2,medial_meniscus_injury,0.555556,0.384615,0.454545,0.567308,26,32
9,bakers_cyst,1.000000,0.250000,0.400000,0.625000,12,46
8,synovitis,0.666667,0.222222,0.333333,0.562724,27,31
6,patellofemoral_osteoarthritis,0.250000,0.142857,0.181818,0.449807,21,37


## 2b. LLM confirmer on 58 labeled studies

Rules stay as the proposal. A small instruct model (**Qwen2.5-1.5B-Instruct**, Apache-2.0) must **agree** before ACL/MCL/OA/synovitis rule-positives are kept. Vetoes are masked (`confidence=0`), not trained as negatives.

High-precision rules (fracture, Baker, effusion, menisci) are not sent to the LLM.

**Kaggle:** the model is already attached at `/kaggle/input/models/qwen-lm/qwen2.5/transformers/1.5b-instruct/1`. Sync `src/`, re-run Setup, then this cell. `LLM source` should print that path — not a HuggingFace hub id.

In [11]:
from rsna_knee.reports import HybridLabeler, LLMLabeler, RuleLabeler
from rsna_knee.reports.llm import DEFAULT_CONFIRM_LABELS, DEFAULT_LLM_MODEL_ID

rpt = cfg.get("reports", {})
confirm_labels = list(rpt.get("confirm_labels") or DEFAULT_CONFIRM_LABELS)
llm = LLMLabeler(
    model_path=rpt.get("llm_model_path"),
    model_id=rpt.get("llm_model_id") or DEFAULT_LLM_MODEL_ID,
    device="auto",
    verbose=True,
)
print(f"LLM source : {llm.describe()}")
print(f"Confirm    : {confirm_labels}")
if llm.source_kind == "hub":
    raise RuntimeError(
        "Resolved a HuggingFace hub id; the kernel would download from huggingface.co. "
        "Attach Qwen/Qwen2.5-1.5B-Instruct via Add Input → Models, restart Jupyter, "
        "sync src/, then re-run Setup."
    )
llm.warmup()
print("LLM ready")

hybrid = HybridLabeler(
    llm_labeler=llm,
    confirm_labels=confirm_labels,
    fill_ambiguous=bool(rpt.get("fill_ambiguous", False)),
    route_threshold=float(rpt.get("route_threshold", 0.7)),
    verbose=True,
)
hybrid_metrics, hybrid_macro_f1 = evaluate_labeler(hybrid, DATA_ROOT)

# Rules eval is cheap; recompute if this cell was run without section 2.
if "rule_metrics" not in globals() or "rule_macro_f1" not in globals():
    rule_metrics, rule_macro_f1 = evaluate_labeler(RuleLabeler(), DATA_ROOT)

print(f"Hybrid macro F1: {hybrid_macro_f1:.3f}  (rules {rule_macro_f1:.3f})")

cmp = rule_metrics[["label", "precision", "recall", "f1"]].merge(
    hybrid_metrics[["label", "precision", "recall", "f1"]],
    on="label",
    suffixes=("_rule", "_hybrid"),
)
cmp["precision_delta"] = cmp["precision_hybrid"] - cmp["precision_rule"]
display(cmp.sort_values("precision_hybrid", ascending=False))
display(hybrid_metrics.sort_values("f1", ascending=False))

LLM source : kaggle:/kaggle/input/models/qwen-lm/qwen2.5/transformers/1.5b-instruct/1
Confirm    : ['acl_tear', 'mcl_tear', 'medial_osteoarthritis', 'lateral_osteoarthritis', 'patellofemoral_osteoarthritis', 'synovitis']


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM ready


...

=== LLM call 28 ===
Asked keys : ['medial_osteoarthritis', 'lateral_osteoarthritis', 'synovitis']
Report (392 chars):
Técnica: RMN de la rodilla. Resultados: Cambios de señal en el menisco interno sin signos de desgarro. Secuelas de osteocondritis de la tuberosidad tibial. OA femorotibial medial y lateral. OA patelofemoral. Derrame con sinovitis. Quiste subcondral en la meseta tibial lateral. Impresión: Cambios de señal en el menisco interno sin signos de desgarro. OA patelofemoral. Derrame con sinovitis.
Raw reply:
```json
{
  "medial_osteoarthritis": 0,
  "lateral_osteoarthritis": 1,
  "synovitis": 1
}
```
Parsed JSON for asked keys: {'medial_osteoarthritis': 0, 'lateral_osteoarthritis': 1, 'synovitis': 1}
=== Match 28 (rule AND llm → keep; veto → mask conf=0) ===
  medial_osteoarthritis: rule=1 llm=0 → veto (label=0, conf=0 masked)
  lateral_osteoarthritis: rule=1 llm=1 → confirm (keep 1)
  synovitis: rule=1 llm=1 → confirm (keep 1)


Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== LLM call 29 ===
Asked keys : ['acl_tear', 'synovitis']
Report (863 chars):
Stigmata der Inaktivitätsosteopenie sowie verbliebenes bone bruise im posterolateralen Tibiaplateau nahe der Eminentia bei subchondraler Impression mit teils punktuell chondralen Läsionen. Anamnestisch bek. VKB Ruptur mit wohl ligamentärer Ruptur des anteromedialen Bündels sowie knöchernem Ausriss des posterolateralen Bündels (4/23). Weiteres Ossikel/Flake angrenzend an die nicht sicher abgrenzbare Außenmeniskushinterhornwurzel, hier DD ebenfalls Mitbeteiligung/knöcherner Ausriss. Innenmeniskus mit horizontalem Riss Pars intermedia bis ins Hinterhorn ziehend bei mukoider Vorschädigung. Zusätzlich noch Knochenödem im medialen femorotibialen Gelenkkompartiment randständig, tibial führend, bei vorrangig Chondropathie. Begleitend geringer Gelenkerguss, Reizsynovialitis sowie gering Flüssigkeit gefüllte Baker-Zyste mit CC 3,2 cm. Kl. Plica mediopatellaris.
Raw reply:
```json
{
  "acl_tear": 0,
  "synovitis": 1
}


,label,precision_rule,recall_rule,f1_rule,precision_hybrid,recall_hybrid,f1_hybrid,precision_delta
9,bakers_cyst,1.000000,0.250000,0.400000,1.000000,0.250000,0.400000,0.000000
11,fracture,1.000000,0.388889,0.560000,1.000000,0.388889,0.560000,0.000000
4,medial_osteoarthritis,1.000000,0.066667,0.125000,1.000000,0.133333,0.235294,0.000000
7,joint_effusion,0.761905,0.457143,0.571429,0.789474,0.428571,0.555556,0.027569
0,acl_tear,0.518519,0.583333,0.549020,0.727273,0.333333,0.457143,0.208754
8,synovitis,0.666667,0.222222,0.333333,0.636364,0.259259,0.368421,-0.030303
3,lateral_meniscus_injury,0.619048,0.565217,0.590909,0.631579,0.521739,0.571429,0.012531
2,medial_meniscus_injury,0.555556,0.384615,0.454545,0.562500,0.346154,0.428571,0.006944
10,bone_contusion,0.529412,0.473684,0.500000,0.529412,0.473684,0.500000,0.000000
1,mcl_tear,0.333333,0.777778,0.466667,0.500000,0.333333,0.400000,0.166667


,label,precision,recall,f1,auc,support_pos,support_neg
3,lateral_meniscus_injury,0.631579,0.521739,0.571429,0.660870,23,35
11,fracture,1.000000,0.388889,0.560000,0.694444,18,40
7,joint_effusion,0.789474,0.428571,0.555556,0.627329,35,23
10,bone_contusion,0.529412,0.473684,0.500000,0.634278,19,39
0,acl_tear,0.727273,0.333333,0.457143,0.622549,24,34
2,medial_meniscus_injury,0.562500,0.346154,0.428571,0.563702,26,32
5,lateral_osteoarthritis,0.500000,0.363636,0.421053,0.639265,11,47
1,mcl_tear,0.500000,0.333333,0.400000,0.636054,9,49
9,bakers_cyst,1.000000,0.250000,0.400000,0.625000,12,46
8,synovitis,0.636364,0.259259,0.368421,0.565114,27,31


## 3. Generate hybrid pseudo-labels

Reuses the confirmer from 2b. The 58 labeled studies still get ground truth.

`/kaggle/working` is wiped when Jupyter restarts. This cell **loads** an existing CSV if present (`/kaggle/working/outputs/` or Dataset `simonhochwebde/rsna-knee-pseudo-labels`, including `/kaggle/input/datasets/...`). Set `FORCE_RELABEL = True` only when you want a new labeling pass.


In [15]:
from rsna_knee.reports.hybrid import load_pseudo_labels

FORCE_RELABEL = False  # True = run the LLM over all 4,407 studies again
_PSEUDO_NAMES = ("pseudo_labels.csv", "pseudo_labels.parquet")
_PSEUDO_MOUNT = Path("/kaggle/input/datasets/simonhochwebde/rsna-knee-pseudo-labels")


def _resolve_pseudo(preferred: Path) -> Path | None:
    """Find CSV/parquet; do not import persist (often missing on a stale code Dataset)."""
    bases = [
        preferred,
        Path("/kaggle/working/outputs") / "pseudo_labels.csv",
        _PSEUDO_MOUNT,
        Path("/kaggle/input/rsna-knee-pseudo-labels"),
    ]
    hits: list[Path] = []
    for base in bases:
        if base.is_file():
            hits.append(base)
            continue
        if not base.is_dir():
            continue
        for name in _PSEUDO_NAMES:
            p = base / name
            if p.is_file():
                hits.append(p)
        try:
            hits.extend(p for p in base.rglob("pseudo_labels.csv") if p.is_file())
            hits.extend(p for p in base.rglob("pseudo_labels.parquet") if p.is_file())
        except OSError:
            continue
    seen: set[str] = set()
    for path in hits:
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        if path.is_file() and path.suffix.lower() in {".csv", ".parquet"}:
            return path
    print("pseudo-label mount:", _PSEUDO_MOUNT, "exists=", _PSEUDO_MOUNT.is_dir())
    if _PSEUDO_MOUNT.is_dir():
        print("contents:", sorted(p.name for p in _PSEUDO_MOUNT.rglob("*"))[:40])
    return None


pseudo_path = Path(cfg["data"]["pseudo_labels_path"])
if not pseudo_path.is_absolute():
    pseudo_path = REPO_ROOT / pseudo_path
pseudo_path.parent.mkdir(parents=True, exist_ok=True)

found = None if FORCE_RELABEL else _resolve_pseudo(pseudo_path)
if found is not None:
    if found.resolve() != pseudo_path.resolve():
        shutil.copy2(found, pseudo_path)
    pseudo_df = load_pseudo_labels(pseudo_path)
    print(f"Loaded {pseudo_path}  shape={pseudo_df.shape}  (from {found})")
elif FORCE_RELABEL:
    llm.verbose = False
    pseudo_df = generate_pseudo_labels(
        DATA_ROOT,
        output_path=pseudo_path,
        llm_labeler=llm,
        confirm_labels=confirm_labels,
        fill_ambiguous=bool(rpt.get("fill_ambiguous", False)),
        route_threshold=float(rpt.get("route_threshold", 0.7)),
    )
    print(f"Wrote {pseudo_path}  shape={pseudo_df.shape}")
else:
    raise FileNotFoundError(
        f"No pseudo_labels.csv under {_PSEUDO_MOUNT}. "
        "If the file has another name, re-run and check the contents listing above."
    )

print(pseudo_df["label_source"].value_counts())
display(pseudo_df.head())


Loaded /kaggle/working/outputs/pseudo_labels.csv  shape=(4407, 26)  (from /kaggle/input/datasets/simonhochwebde/rsna-knee-pseudo-labels/pseudo_labels.csv)
label_source
rules           3035
hybrid          1314
ground_truth      58
Name: count, dtype: int64


,StudyInstanceUID,label_source,acl_tear,acl_tear_conf,mcl_tear,mcl_tear_conf,medial_meniscus_injury,medial_meniscus_injury_conf,lateral_meniscus_injury,lateral_meniscus_injury_conf,...,joint_effusion,joint_effusion_conf,synovitis,synovitis_conf,bakers_cyst,bakers_cyst_conf,bone_contusion,bone_contusion_conf,fracture,fracture_conf
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,hybrid,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,...,1.0,0.90,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00
1,1.2.826.0.1.3680043.8.498.10004945927472656027...,rules,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,...,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00
2,1.2.826.0.1.3680043.8.498.10009278692606631573...,hybrid,0.0,0.0,0.0,0.0,0.0,0.85,0.0,0.85,...,0.0,0.85,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00
3,1.2.826.0.1.3680043.8.498.10009639203170750274...,rules,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.85,...,0.0,0.85,0.0,0.0,0.0,0.0,0.0,0.85,0.0,0.85
4,1.2.826.0.1.3680043.8.498.10013663742400736029...,rules,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,...,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00


## 3b. Keep the CSV on your PC

Kernel Dataset upload is skipped. The file stays at `/kaggle/working/outputs/pseudo_labels.csv` until this Jupyter session ends.

On your PC (session still running):

```bash
python scripts/pull_pseudo_labels.py
```

That writes `outputs/pseudo_labels.csv` (gitignored). Upload later:

```bash
python scripts/publish_pseudo_labels.py
```

In [11]:
preferred = pseudo_path if "pseudo_path" in dir() else Path("/kaggle/working/outputs/pseudo_labels.csv")
found = _resolve_pseudo(preferred) if "_resolve_pseudo" in dir() else (preferred if preferred.is_file() else None)
print("On Kaggle:", found)
if found is not None:
    print(f"size={found.stat().st_size / 1e6:.2f} MB  rows={sum(1 for _ in found.open()) - 1}")
print("Pull to PC while this session is up:  python scripts/pull_pseudo_labels.py")
print("Upload Dataset later:                 python scripts/publish_pseudo_labels.py")

On Kaggle: None
Pull to PC while this session is up:  python scripts/pull_pseudo_labels.py
Upload Dataset later:                 python scripts/publish_pseudo_labels.py


## 4. Resolve pretrained weights

In [13]:
weight_name = cfg["model"].get("pretrained_weights", "convnext_tiny_imagenet.pth")
variant = cfg["model"].get("pretrained_variant")
pretrained_path = resolve_pretrained_weights(
    filename=weight_name,
    variant=variant,
    allow_missing=bool(cfg["model"].get("allow_random_init", False)),
)
print(f"Pretrained weights: {pretrained_path}")

Pretrained weights: /kaggle/input/datasets/simonhochwebde/rsna-knee-pretrained/convnext_tiny_imagenet.pth


## 5a. Pack volume cache (local only)

Decodes DICOMs into `/kaggle/working/volume_cache` (~14 GiB). **Does not upload.**

If `d16_h256_w256_s3` already has 4407 `.npy` files, skip this cell and run the four **Upload part** cells below.

Internet ON for uploads only (parts 1-4).


In [ ]:
import gc

from rsna_knee.data import (
    materialize_writable_volume_cache,
    prepare_disk_volume_cache,
)

if "llm" in dir():
    unload = getattr(llm, "unload", None)
    if callable(unload):
        unload()
    else:
        del llm
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print("Released LLM weights")

CACHE_DIR = Path("/kaggle/working/volume_cache") if ON_KAGGLE else REPO_ROOT / "volume_cache"
volume_shape = tuple(cfg["data"]["volume_shape"])
max_series = int(cfg["data"]["max_series"])
_pseudo = pseudo_path if "pseudo_path" in dir() else cfg["data"].get("pseudo_labels_path")

cache_ds = prepare_disk_volume_cache(
    DATA_ROOT,
    CACHE_DIR,
    labeled_only=bool(cfg["data"].get("labeled_only", False)),
    pseudo_labels_path=_pseudo,
    min_confidence=float(cfg["data"].get("min_confidence", 0.7)),
    volume_shape=volume_shape,
    max_series=max_series,
    max_workers=8,
)
n_copied = materialize_writable_volume_cache(cache_ds)
n_npy = len(list(cache_ds.cache_dir.glob("*.npy")))
print(f"Cache ready: {cache_ds.cache_dir}  npy={n_npy}  copied_from_mounts={n_copied}")
print("Next: run Upload part 1, then 2, then 3 (internet ON).")


## 5a. Upload part 1 / 4 (~3 GiB)

Internet **ON**. Run this cell alone; wait until it finishes before part 2.

Creates Dataset `simonhochwebde/rsna-knee-volume-cache-p1`.


In [21]:
from pathlib import Path

from rsna_knee.data.volume_cache import push_volume_cache_part

CACHE_DIR = Path("/kaggle/working/volume_cache/d16_h256_w256_s3")
print("cache:", CACHE_DIR)
push_volume_cache_part(CACHE_DIR, 1, n_parts=4)
print("Part 1/4 done. Run Upload part 2.")


ImportError: cannot import name 'push_volume_cache_part' from 'rsna_knee.data.volume_cache' (/kaggle/working/src/rsna_knee/data/volume_cache.py)

## 5a. Upload part 2 / 4


In [ ]:
from pathlib import Path

from rsna_knee.data.volume_cache import push_volume_cache_part

CACHE_DIR = Path("/kaggle/working/volume_cache/d16_h256_w256_s3")
print("cache:", CACHE_DIR)
push_volume_cache_part(CACHE_DIR, 2, n_parts=4)
print("Part 2/4 done. Run Upload part 3.")


## 5a. Upload part 3 / 4


In [ ]:
from pathlib import Path

from rsna_knee.data.volume_cache import push_volume_cache_part

CACHE_DIR = Path("/kaggle/working/volume_cache/d16_h256_w256_s3")
print("cache:", CACHE_DIR)
push_volume_cache_part(CACHE_DIR, 3, n_parts=4)
print("Part 3/4 done. Run Upload part 4.")


## 5a. Upload part 4 / 4

After success: attach `rsna-knee-volume-cache-p1`, `-p2`, `-p3`, `-p4` on the kernel, restart Jupyter, then run **5b Train**.

In [ ]:
from pathlib import Path

from rsna_knee.data.volume_cache import push_volume_cache_part

CACHE_DIR = Path("/kaggle/working/volume_cache/d16_h256_w256_s3")
print("cache:", CACHE_DIR)
push_volume_cache_part(CACHE_DIR, 4, n_parts=4)
print("Part 4/4 done. Attach all four Datasets and restart Jupyter.")

## 5b. Train on pseudo-labels (OOF on 58 labeled)

Reads packed volumes from `/kaggle/working/volume_cache` and/or Datasets `rsna-knee-volume-cache-p1` … `-p4`. Does not re-decode DICOMs. Prefer GPU (T4), internet OFF.


In [ ]:
import inspect

from rsna_knee.training import loop as _train_loop

CKPT_DIR = Path(cfg["paths"]["checkpoint_dir"])
if not CKPT_DIR.is_absolute():
    CKPT_DIR = REPO_ROOT / CKPT_DIR
CKPT_DIR.mkdir(parents=True, exist_ok=True)

volume_shape = tuple(cfg["data"]["volume_shape"])
max_series = int(cfg["data"]["max_series"])
MAX_EPOCHS = int(cfg["training"]["max_epochs"])
if DEVICE.type == "cpu":
    MAX_EPOCHS = min(MAX_EPOCHS, 1)

if "CACHE_DIR" not in dir():
    CACHE_DIR = Path(cfg["data"]["cache_dir"] or "/kaggle/working/volume_cache")

print("loop.py:", _train_loop.__file__)
print("cache_dir:", CACHE_DIR)
_kfold_params = inspect.signature(_train_loop.run_kfold_training).parameters
train_kw = dict(
    n_folds=int(cfg["training"]["n_folds"]),
    max_epochs=MAX_EPOCHS,
    batch_size=int(cfg["training"]["batch_size"]),
    learning_rate=float(cfg["training"]["learning_rate"]),
    volume_shape=volume_shape,
    max_series=max_series,
    checkpoint_dir=CKPT_DIR,
    seed=int(cfg["seed"]),
    pretrained_path=pretrained_path,
    allow_random_init=bool(cfg["model"].get("allow_random_init", False)),
    tta=bool(cfg["inference"].get("tta", False)),
    num_workers=int(cfg["data"].get("num_workers", 0)),
    model_name=cfg["model"]["name"],
    use_amp=bool(cfg["training"].get("amp", True)),
    gpu_cache=bool(cfg["training"].get("gpu_cache", False)),
    data_parallel=bool(cfg["training"].get("data_parallel", False)),
    parallel_folds=cfg["training"].get("parallel_folds"),
    slice_chunk=int(cfg["model"].get("slice_chunk", 16)),
    labeled_only=bool(cfg["data"].get("labeled_only", False)),
    pseudo_labels_path=pseudo_path if "pseudo_path" in dir() else cfg["data"].get("pseudo_labels_path"),
    min_confidence=float(cfg["data"].get("min_confidence", 0.7)),
    confidence_weighted_loss=bool(cfg["training"].get("confidence_weighted_loss", True)),
    eval_labeled_only=bool(cfg["training"].get("eval_labeled_only", True)),
    mixup_alpha=float(cfg["training"].get("mixup_alpha", 0.4)),
    use_multimodal=bool(cfg["model"].get("use_multimodal", False)),
    text_model_path=cfg["model"].get("text_model_path"),
    cache_dir=str(CACHE_DIR),
    warm_cache=False,
)
dropped = [k for k in train_kw if k not in _kfold_params]
train_kw = {k: v for k, v in train_kw.items() if k in _kfold_params}
if dropped:
    print("Dropped kwargs not in this loop.py:", dropped)

train_result = run_phase2_training(DATA_ROOT, **train_kw)

print(f"OOF macro ROC-AUC (58 labeled): {train_result['overall_auc']:.4f}")
print("Phase 1 baseline (from-scratch, 58 only): 0.5295")


## 6. Per-label OOF detail

In [ ]:
train = load_train_table(DATA_ROOT)
labeled_mask = labels_present_mask(train)
labeled_ids = set(train.loc[labeled_mask, "StudyInstanceUID"].astype(str))
labeled_idx = [i for i, uid in enumerate(train_result["study_ids"]) if uid in labeled_ids]

oof_preds = train_result["oof_preds"][labeled_idx]
oof_labels = train_result["oof_labels"][labeled_idx]
print("OOF macro ROC-AUC:", macro_roc_auc(oof_labels, oof_preds))

from sklearn.metrics import roc_auc_score
rows = []
for i, name in enumerate(TARGET_LABELS):
    yt = oof_labels[:, i]
    if len(np.unique(yt)) < 2:
        rows.append({"label": name, "auc": float("nan")})
        continue
    rows.append({"label": name, "auc": float(roc_auc_score(yt, oof_preds[:, i]))})
display(pd.DataFrame(rows).sort_values("auc", ascending=False))

## 7. Test inference → submission.csv (image-only)

In [ ]:
ckpt_paths = [fr.checkpoint_path for fr in train_result["fold_results"]]
study_ids, test_preds = predict_test_ensemble(
    ckpt_paths,
    DATA_ROOT,
    volume_shape=volume_shape,
    max_series=max_series,
    batch_size=int(cfg["training"]["batch_size"]),
    tta=bool(cfg["inference"].get("tta", False)),
    allow_random_init=True,
    model_name=cfg["model"]["name"],
    num_workers=int(cfg["data"].get("num_workers", 0)),
    use_amp=bool(cfg["training"].get("amp", True)),
    slice_chunk=int(cfg["model"].get("slice_chunk", 16)),
)

sub = predictions_to_submission(study_ids, pd.DataFrame(test_preds, columns=TARGET_LABELS))
sub_path = OUT_DIR / "submission.csv"
sub.to_csv(sub_path, index=False)
print(f"Wrote {sub_path}  shape={sub.shape}")
display(sub.head())

## 8. Optional — multimodal experiment

Set `model.use_multimodal: true` and attach a clinical text encoder Dataset. Re-run section 5 with fusion at train time; inference remains image-only.